# Proceso de automatizacion contable

In [1]:
# importar librerias a usar
import pandas as pd
from clase_reportes import ReportClass
import numpy as np
import json
rc = ReportClass()


In [2]:
ruta = rc.validar_ruta()
ruta_contabilidad = ruta / 'data' / 'contabilidad'
df_4 = rc.consolidar_carpeta(ruta_carpeta=ruta_contabilidad / '4' )
df_5 = rc.consolidar_carpeta(ruta_carpeta=ruta_contabilidad / '5' )
df_6 = rc.consolidar_carpeta(ruta_carpeta=ruta_contabilidad / '6' )
df_base = pd.concat([df_4,df_5,df_6], ignore_index=True)
df_base = df_base.rename(columns={'Cuenta': 'Cuenta Origen'})
df_base['Cuenta'] = df_base['Cuenta Origen'].str.split(' ', regex=True).str[0]
df_base['Nombre cuenta'] = df_base['Cuenta Origen'].str.split(' ', regex=True).str[1:].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')
df_base['N1'] = df_base['Cuenta Origen'].astype(str).str[0]
df_base['N2'] = df_base['Cuenta Origen'].astype(str).str[:2]
df_base['N3'] = df_base['Cuenta Origen'].astype(str).str[:4]

# df_base['Débito'] = df_base['Débito'].fillna(0)
# df_base['Crédito'] = df_base['Crédito'].fillna(0)
# df_base['Balance'] = df_base['Débito'] - df_base['Crédito']

# Define la columna nivel
df_base['Nivel']  =np.where(df_base['N2']==41, 'Ingreso Operativo',
         np.where(df_base['N2']==42, 'Otros ingresos',
                  np.where(df_base['N2']==52, 'Gastos operacionales',
                           np.where(df_base['N2']==53, 'Gastos No Operacionales',
                                    np.where(df_base['N2']==61, 'Costo directo de ventas', 
                                             'Revisar'
                                    )
                           )
                  )
         )
)

df_niveles =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='niveles')
df_detalle =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='detalle')
df_concepto =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='concepto')
df_cc =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='CC')
df_base['N3'] = df_base['N3'].astype(int)
df_base['N3'] = df_base['N3'].astype(int)
df_base['Cuenta'] = df_base['Cuenta'].astype(int)

df_base_merge = df_base.merge(df_niveles, left_on='N3', right_on='cuenta', how='left').drop(columns='cuenta')

df_base_merge = df_base_merge.merge(df_detalle, left_on='Cuenta', right_on='CUENTA', how='left').drop(columns=['CUENTA', 'NOMBRE'])


# Supongamos que la columna se llama 'columna_dict'
def extraer_clave(diccionario_str):
    if pd.isna(diccionario_str):
        return None
    try:
        diccionario = json.loads(diccionario_str)
        return list(diccionario.keys())[0]
    except Exception:
        return None

df_base_merge = df_base_merge.rename(columns={'Distribución analítica': 'Distribución analítica ori'})

df_base_merge['Distribución analítica'] = df_base_merge['Distribución analítica ori'].apply(extraer_clave)
# Centros de costo mal clasificados
cc_corregir = df_base_merge[df_base_merge['Distribución analítica ori'].fillna('').str.count(':')>1]
# Genera el archivo de los casos sin centro de costos
sin_cc = df_base_merge[df_base_merge['Distribución analítica'].isna()]
sin_cc.to_excel(ruta_contabilidad / 'sin_cc.xlsx', index=False)
# Guarda las errores en un solo archivo
with pd.ExcelWriter(ruta_contabilidad / 'correciones.xlsx', engine='openpyxl') as writer:
    sin_cc.to_excel(writer, index=False, sheet_name='Sin CC')
    cc_corregir.to_excel(writer, index=False, sheet_name='Corregir CC')

df_cc['cc'] = df_cc['cc'].astype(str)
df_base_merge = df_base_merge.merge(df_cc[['cc','NOMBRE CENTRO COSTOS', 'ADM/VTAS' ]],
                                     left_on='Distribución analítica', right_on='cc', how='left').drop(columns='cc')
df_concepto['CC'] = df_concepto['CC'].str.upper().str.strip()
df_base_merge['NOMBRE CENTRO COSTOS'] = df_base_merge['NOMBRE CENTRO COSTOS'].str.upper().str.strip()
df_base_merge = df_base_merge.merge(df_concepto, left_on=['Cuenta','NOMBRE CENTRO COSTOS' ], right_on=['Cuenta','CC'], how='left')


Buscando archivos con extensión '.xlsx' en: C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\data\contabilidad\4
  - Archivo 'Apunte contable (account.move.line).xlsx' leído correctamente.
Concatenando todos los archivos...
¡Consolidación completada!
Buscando archivos con extensión '.xlsx' en: C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\data\contabilidad\5
  - Archivo 'Apunte contable (account.move.line) (1).xlsx' leído correctamente.
Concatenando todos los archivos...
¡Consolidación completada!
Buscando archivos con extensión '.xlsx' en: C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\data\contabilidad\6
  - Archivo 'Apunte contable (account.move.line) (2).xlsx' leído correctamente.
Concatenando todos los archivos...
¡Consolidación completada!


In [55]:
df_base_merge.to_csv(r"C:\Users\Dataa\Desktop\base_pyg.csv", sep=";", index=False, encoding='utf-8')